# STEAM VIDEO GAME RECOMMENDER SYSTEM USING ALTERNATING LEAST SQUARES (ALS)

In this notebook, we will build a recommender system for video games using data collected from Steam gaming platform. The algorithm that will be used for building the recommender system is Alternating Least Squares (ALS), a collaborative filtering method.

## Setup
___

### Import Necessary Libraries

In [80]:
import mlflow
import logging
import random
import numpy as np
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.ml.evaluation import RegressionEvaluator, RankingEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

from utils.logger import logger

In [81]:
# Start MLflow server (Run this in a separate terminal before executing the notebook)
# mlflow server \
#   --backend-store-uri sqlite:///mlflow.db \
#   --default-artifact-root ./mlartifacts \
#   --host 127.0.0.1 \
#   --port 5000

In [82]:
# Create Spark session
spark = SparkSession.builder.appName("SteamRecommenderSystem").master("local[*]").getOrCreate()

### Configure MLFlow for Experiment Tracking

In [83]:
logging.getLogger("mlflow").setLevel(logging.ERROR)
mlflow.pyspark.ml.autolog()

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("local-uc")

experiment_name = "steam_recommender_system"

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name)
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1780136948096, experiment_id='1', last_update_time=1780136948096, lifecycle_stage='active', name='steam_recommender_system', tags={}, trace_location=None, workspace='default'>

In [84]:
# Set random seed value for reproducibility
SEED = 100

In [85]:
# Download the dataset from Kaggle (Run this only once to download the dataset)
!curl -L -o data/steam-video-games.zip\
  https://www.kaggle.com/api/v1/datasets/download/tamber/steam-video-games

IOStream.flush timed out
IOStream.flush timed out
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1497k  100 1497k    0     0  1514k      0 --:--:-- --:--:-- --:--:-- 10.8M


In [86]:
# Unzip the downloaded dataset
!unzip -o data/steam-video-games.zip -d data/

IOStream.flush timed out
IOStream.flush timed out
Archive:  data/steam-video-games.zip
  inflating: data/steam-200k.csv     


### Load Dataset

Before loading the dataset, we need to inspect the structure of the data; this will inform us on the proper approach to take when loading the data into a dataframe. To do this, we use the linux command `head` to display the first few lines of our dataset.

In [87]:
!head -n 20 data/steam-200k.csv

IOStream.flush timed out
IOStream.flush timed out
151603712,"The Elder Scrolls V Skyrim",purchase,1.0,0
151603712,"The Elder Scrolls V Skyrim",play,273.0,0
151603712,"Fallout 4",purchase,1.0,0
151603712,"Fallout 4",play,87.0,0
151603712,"Spore",purchase,1.0,0
151603712,"Spore",play,14.9,0
151603712,"Fallout New Vegas",purchase,1.0,0
151603712,"Fallout New Vegas",play,12.1,0
151603712,"Left 4 Dead 2",purchase,1.0,0
151603712,"Left 4 Dead 2",play,8.9,0
151603712,"HuniePop",purchase,1.0,0
151603712,"HuniePop",play,8.5,0
151603712,"Path of Exile",purchase,1.0,0
151603712,"Path of Exile",play,8.1,0
151603712,"Poly Bridge",purchase,1.0,0
151603712,"Poly Bridge",play,7.5,0
151603712,"Left 4 Dead",purchase,1.0,0
151603712,"Left 4 Dead",play,3.3,0
151603712,"Team Fortress 2",purchase,1.0,0
151603712,"Team Fortress 2",play,2.8,0


Observations: From the output, we can confirm that:
- The dataset has no header, so we will need to set the `header` argument in `spark.read.csv` to `False`, and set out headers using the `toDF` method.
- The columns are separated by commas, so our delimiter, the `sep` argument, needs to be set to `,`, which is already the default value

In [88]:
steam_data_raw = spark.read.csv("data/steam-200k.csv",
                         header=False,
                         inferSchema=True) \
                      .toDF("user_id", "game_name", "behaviour", "purchase_playtime", "unknown")

### Dataset Description
- `user_id`: contains a unique identifier for each member
- `game_name`: contains the name of the game they purchased or played
- `behaviour`: contains details of the member behaviour, either `purchase` or `play`
- `purchase_playtime`: is set to `1` for rows where the `behaviour` is `purchase`, and number of hours played for rows where the `behaviour` is set to `play`
- `unknown`: contains `0` for every row, and is not useful for our analysis. We will drop this column in the next step.

## Initial Exploratory Data Analysis (EDA)
___
In this section, we will be conducting initial exploratory data analysis to understand our raw data to inform us on the appropriate data preprocessing steps to take.

In [89]:
# Show the schema

steam_data_raw.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- game_name: string (nullable = true)
 |-- behaviour: string (nullable = true)
 |-- purchase_playtime: double (nullable = true)
 |-- unknown: integer (nullable = true)



In [90]:
steam_data_raw.show(5)

[2026-05-30 13:07:50.713964] INFO - Error while sending or receiving. (/media/vince/Backup/Projects/steam_video_game_recommender_sys/.venv/lib/python3.11/site-packages/py4j/clientserver.py:529:send_command())
[2026-05-30 13:07:50.715075] INFO - Closing down clientserver connection (/media/vince/Backup/Projects/steam_video_game_recommender_sys/.venv/lib/python3.11/site-packages/py4j/clientserver.py:570:close())
[2026-05-30 13:07:50.716306] INFO - Exception while sending command. (/media/vince/Backup/Projects/steam_video_game_recommender_sys/.venv/lib/python3.11/site-packages/py4j/java_gateway.py:1056:send_command())
[2026-05-30 13:07:50.720228] INFO - Closing down clientserver connection (/media/vince/Backup/Projects/steam_video_game_recommender_sys/.venv/lib/python3.11/site-packages/py4j/clientserver.py:570:close())


+---------+--------------------+---------+-----------------+-------+
|  user_id|           game_name|behaviour|purchase_playtime|unknown|
+---------+--------------------+---------+-----------------+-------+
|151603712|The Elder Scrolls...| purchase|              1.0|      0|
|151603712|The Elder Scrolls...|     play|            273.0|      0|
|151603712|           Fallout 4| purchase|              1.0|      0|
|151603712|           Fallout 4|     play|             87.0|      0|
|151603712|               Spore| purchase|              1.0|      0|
+---------+--------------------+---------+-----------------+-------+
only showing top 5 rows


In [91]:
steam_data_raw = steam_data_raw.drop("unknown")

**Observations:** 
- From inspecting a subset of the dataset, we can see that there are two entries for each `user_id`/`game_name` pair. This is because a game needs to be purchased before it can be played.
- The `purchase_playtime` column is set to `1`where the behaviour is `purchase`.
- We are missing unique integer identifiers for the game, and to perform ALS, we will need these integers. The unique identifiers will be generated in the [feature engineering](#feature-engineering) section of this notebook.

In [92]:
# Check the number of missing values in each row

steam_data_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in steam_data_raw.columns
]).show()

+-------+---------+---------+-----------------+
|user_id|game_name|behaviour|purchase_playtime|
+-------+---------+---------+-----------------+
|      0|        0|        0|                0|
+-------+---------+---------+-----------------+



Let's verify the behaviours we have in our dataset

In [93]:
# Show the distinct behaviours

steam_data_raw.select("behaviour").distinct().show()

+---------+
|behaviour|
+---------+
| purchase|
|     play|
+---------+



In [94]:
# Show the number of purchases
purchase_count = steam_data_raw.filter(F.col("behaviour") == "purchase").count()
logger.info(f"Number of purchases: {purchase_count}")

[2026-05-30 13:07:51.869912] INFO - Number of purchases: 129511 (/tmp/ipykernel_805233/2719036694.py:3:<module>())


In [95]:
# Show number of plays
play_count = steam_data_raw.filter(F.col("behaviour") == "play").count()
logger.info(f"Number of plays: {play_count}")

[2026-05-30 13:07:52.057964] INFO - Number of plays: 70489 (/tmp/ipykernel_805233/1632857843.py:3:<module>())


In [96]:
# Show percentage of games purchased that were played
logger.info(f"Percentage plays: {play_count / (purchase_count):.2%}")

[2026-05-30 13:07:52.088533] INFO - Percentage plays: 54.43% (/tmp/ipykernel_805233/3212893287.py:2:<module>())


**Observations:**
- Only `54.43%` of the games that were purchased on the Steam platform was actually played

In [97]:

steam_data_raw.filter(F.col("behaviour") == "purchase").select("purchase_playtime").distinct().show()

+-----------------+
|purchase_playtime|
+-----------------+
|              1.0|
+-----------------+



In [98]:
# Show the number of distinct users

distinct_users = steam_data_raw.select("user_id").distinct()

logger.info(f"Number of distinct users: {distinct_users.count()}")

[2026-05-30 13:07:52.807261] INFO - Number of distinct users: 12393 (/tmp/ipykernel_805233/1611889130.py:5:<module>())


In [99]:
# Show the number of distinct games

distinct_games = steam_data_raw.select("game_name").distinct()

logger.info(f"Number of distinct games: {distinct_games.count()}")

[2026-05-30 13:07:53.055150] INFO - Number of distinct games: 5155 (/tmp/ipykernel_805233/3839261409.py:5:<module>())


**Observation:**
- We can see that there is a discrepancy in the value of your unique games played. In our `Raw Data Profile` generated in Cell 18, the unique count of our categorical feature `game_name` is `5348`, however, when using distinct on `game_name` column, the unique count is shown to be `5155`. Databricks also noted that the data generated in `Raw Data Profile` may be an approximation. To be on the safe side, we will normalize the `game_name` column and then compare the unique count after normalization with that previous distinct value count.

In [100]:
# Visualize the distribution of playtime
playtime = (
    steam_data_raw
    .filter(F.col("behaviour") == "play")
    .select("purchase_playtime")
)
playtime.show()

+-----------------+
|purchase_playtime|
+-----------------+
|            273.0|
|             87.0|
|             14.9|
|             12.1|
|              8.9|
|              8.5|
|              8.1|
|              7.5|
|              3.3|
|              2.8|
|              2.5|
|              2.0|
|              1.4|
|              1.3|
|              1.3|
|              0.8|
|              0.8|
|              0.6|
|              0.5|
|              0.5|
+-----------------+
only showing top 20 rows


In [101]:
# Show the minimum and maximum playtime

min_playtime = playtime.agg(F.min("purchase_playtime")).collect()[0][0]
max_playtime = playtime.agg(F.max("purchase_playtime")).collect()[0][0]

logger.info(f"Minimum playtime: {min_playtime} hours")
logger.info(f"Maximum playtime: {max_playtime} hours")

[2026-05-30 13:07:53.467841] INFO - Minimum playtime: 0.1 hours (/tmp/ipykernel_805233/129657257.py:6:<module>())
[2026-05-30 13:07:53.468699] INFO - Maximum playtime: 11754.0 hours (/tmp/ipykernel_805233/129657257.py:7:<module>())


In [102]:
# Show the average playtime

avg_playtime = playtime.agg(F.avg("purchase_playtime")).collect()[0][0]

logger.info(f"Average playtime: {avg_playtime:.2f} hours")

[2026-05-30 13:07:53.652235] INFO - Average playtime: 48.88 hours (/tmp/ipykernel_805233/4191913506.py:5:<module>())


In [103]:
playtime.describe().show()

+-------+------------------+
|summary| purchase_playtime|
+-------+------------------+
|  count|             70489|
|   mean| 48.87806324391019|
| stddev|229.33523599681237|
|    min|               0.1|
|    max|           11754.0|
+-------+------------------+



In [104]:

# Show the number of games purchased by each user

user_purchase_count = (
    steam_data_raw
    .filter(F.col("behaviour") == "purchase")
    .groupBy("user_id")
    .count()
    .withColumnRenamed("count", "purchase_count")
    .orderBy("purchase_count", ascending=False))
user_purchase_count.show()

+---------+--------------+
|  user_id|purchase_count|
+---------+--------------+
| 62990992|          1075|
| 33865373|           783|
| 30246419|           766|
| 58345543|           667|
| 76892907|           597|
| 20772968|           595|
| 11403772|           592|
| 64787956|           591|
| 22301321|           568|
| 47457723|           557|
| 33013552|           520|
| 53875128|           505|
| 86469479|           480|
| 49893565|           477|
| 11373749|           458|
| 24721232|           456|
|138941587|           448|
| 36546868|           430|
| 59825286|           428|
|122026623|           419|
+---------+--------------+
only showing top 20 rows


In [105]:
# Show the number of games played by each user

user_play_count = (
    steam_data_raw
    .filter(F.col("behaviour") == "play")
    .groupBy("user_id")
    .count()
    .withColumnRenamed("count","play_count")
    .orderBy("play_count", ascending=False))
user_play_count.show()

+---------+----------+
|  user_id|play_count|
+---------+----------+
| 62990992|       498|
| 11403772|       314|
|138941587|       299|
| 47457723|       298|
| 49893565|       297|
| 24469287|       284|
| 48798067|       254|
| 36546868|       235|
| 51557405|       210|
| 17530772|       209|
|116876958|       208|
| 22301321|       207|
| 11373749|       204|
|   975449|       202|
| 33013552|       200|
| 38049880|       200|
| 53875128|       197|
| 10599862|       178|
| 26762388|       178|
|   298950|       175|
+---------+----------+
only showing top 20 rows


In [106]:
# Join user_purchase_count and user_play_count on user_id

user_play_purchase_count = user_purchase_count.join(user_play_count, "user_id")
user_play_purchase_count.show()

+---------+--------------+----------+
|  user_id|purchase_count|play_count|
+---------+--------------+----------+
| 16167221|            32|        25|
|166705920|            10|         1|
|244878837|             1|         1|
| 99992274|             1|         1|
|174415183|             7|         3|
|156156544|             1|         1|
|152861732|            22|        12|
|171911285|             1|         1|
|128412180|             1|         1|
| 74557142|             3|         1|
|197279094|             3|         3|
|108219790|             2|         2|
|208061820|             6|         5|
| 61502266|             1|         1|
|138455695|             1|         1|
| 35264447|            10|         4|
|187877855|             6|         6|
|148546383|             1|         1|
|119310413|            35|        27|
|261857176|             2|         1|
+---------+--------------+----------+
only showing top 20 rows


Next we need to calculate the sparsity of our dataset

**Calculate the Sparsity of our Dataset**

Sparsity measures how much of our user-item matrix contains missing values (zero/null). The formula is given by:

$$Sparsity = 1 - \frac{\text{Non-Zero Values}}{\text{Total Users} \times \text{Total Items}}$$


In [107]:
# Get the number of interactions. The minimum interaction here is taken as the purchase behaviour
num_interactions = steam_data_raw.filter(F.col("behaviour") == "purchase").count()

# Get the number of users and games
num_users = distinct_users.count()
num_games = distinct_games.count()

# Calculate the sparsity
sparsity = 1 - (num_interactions / (num_users * num_games))
logger.info(f"Sparsity: {sparsity:.4f}")

[2026-05-30 13:07:55.343899] INFO - Sparsity: 0.9980 (/tmp/ipykernel_805233/99369732.py:10:<module>())


**Observations:**
- The sparsity of our user-item matrix is over _99%_, which is typical for recommender systems in production. This is because users interact with a very small percentage of the available items.

## Preprocessing
___

### Normalizing the `game_name` column

In [108]:
normalized_data = (
    steam_data_raw
              .withColumn("game_name", F.lower("game_name"))
              .withColumn("game_name", F.regexp_replace("game_name", r"\s+", "_"))
              .withColumn("game_name", F.trim("game_name"))
              )

In [109]:
# Show the number of distinct games

normalized_distinct_games = normalized_data.select("game_name").distinct()

logger.info(f"Number of distinct games: {normalized_distinct_games.count()}")

[2026-05-30 13:07:55.678365] INFO - Number of distinct games: 5155 (/tmp/ipykernel_805233/255629938.py:5:<module>())


## Feature Engineering
___

### Computing Relevance Score for `user_id` and `game_name` Pair
In order to prepare our dataset for training the ALS model, we need to define an appropriate relevance score that captures the importance of the _**implicit feedback**_. We have two candidate features to work with: purchase and play time. As previously observed in cells 23 and 25, the play time of users varies from `0.1 hours` to `11754.0 hours`, with the mean playtime being `48.88 hours`. This shows extreme skewness in playtime and this will make the model think only really high-hour games matter. We can mitigate the effect of the skewness by rescaling the playtime using the natural logarithm. The following formula will be applied to the playtime:

$$Playtime_{rescaled} = \ln(1 + \textit{hours played})$$

**What does this do?**

| Hours | ln(1 + hours) |
| ----- | -------------- |
| 1     | 0.69           |
| 10    | 2.40           |
| 100   | 4.61           |
| 500   | 6.22           |

As seen in the table above, extremely large values are compressed while still preserving order.

**Why plus 1?**

This is because the natural log of zero is undefined, so to avoid this, a simple trick is to add one to the value.

We also want to utilize the signal from the purchase feature. If a user has purchased a game, we add 1 to the relevance score as a base signal, even if they have not played it yet. Our final relevance score formula becomes:

$$\text{Relevance Score} = 1 + Playtime_{rescaled}$$

The `+1` accounts for the purchase signal, ensuring that purchased but unplayed games still carry some relevance.

In [110]:
processed_data = (
    normalized_data

    # Apply log transformation to purchase_playtime, if behaviour is "purchase", set relevance_score to purchase_playtime (1)
    # Otherwise, set relevance_score to log(1 + purchase_playtime)
    .withColumn(
        "relevance_score", 
        F.when(F.col("behaviour") == "purchase", F.col("purchase_playtime")).otherwise(F.log(1 + F.col("purchase_playtime")))
        )
    
    # Group by user_id and game_name and sum relevance_score
    .groupBy("user_id", "game_name").agg(F.sum("relevance_score").alias("relevance_score"))
    )

In [111]:
# Display first 100 rows
processed_data.show(100)

+---------+--------------------+------------------+
|  user_id|           game_name|   relevance_score|
+---------+--------------------+------------------+
|197455089|              dota_2|2.4816045409242156|
|126340495|             outlast|2.6292405397302803|
| 97298878|        race_the_sun|3.4423470353692043|
| 92107940|far_cry_3_blood_d...| 4.433987204485146|
| 11373749|     team_fortress_2| 7.637258031284457|
| 11373749|             anodyne|3.0412203288596382|
| 11373749|resident_evil_5_/...|2.2237754316221157|
| 11373749|the_movies_stunts...|               1.0|
| 11373749|  hitman_blood_money|               1.0|
|171847029|arma_cold_war_ass...| 1.262364264467491|
| 64350600|half-life_blue_shift|               1.0|
| 56038151|           sanctum_2|               1.0|
| 94088853|dark_souls_prepar...|1.4700036292457357|
|170095814|               smite|               1.0|
|148510973|half-life_blue_shift|               1.0|
|  9823354|           anno_2070| 4.433987204485146|
|  9823354|a

In [112]:
# Display summary statistics
processed_data.select("relevance_score").describe().show()

+-------+------------------+
|summary|   relevance_score|
+-------+------------------+
|  count|            128804|
|   mean| 2.128977130214401|
| stddev|1.5729015986643116|
|    min|               1.0|
|    max| 10.37203396097417|
+-------+------------------+



In [113]:
processed_data.show()

+---------+--------------------+------------------+
|  user_id|           game_name|   relevance_score|
+---------+--------------------+------------------+
|197455089|              dota_2|2.4816045409242156|
|126340495|             outlast|2.6292405397302803|
| 97298878|        race_the_sun|3.4423470353692043|
| 92107940|far_cry_3_blood_d...| 4.433987204485146|
| 11373749|     team_fortress_2| 7.637258031284457|
| 11373749|             anodyne|3.0412203288596382|
| 11373749|resident_evil_5_/...|2.2237754316221157|
| 11373749|the_movies_stunts...|               1.0|
| 11373749|  hitman_blood_money|               1.0|
|171847029|arma_cold_war_ass...| 1.262364264467491|
| 64350600|half-life_blue_shift|               1.0|
| 56038151|           sanctum_2|               1.0|
| 94088853|dark_souls_prepar...|1.4700036292457357|
|170095814|               smite|               1.0|
|148510973|half-life_blue_shift|               1.0|
|  9823354|           anno_2070| 4.433987204485146|
|  9823354|a

Observation:
- The relevance score now scales between `1` and `10.37`, which means our model will now learn to give more importance to games that are purchased and played, while still giving appropriate relevance to games that are only purchased.

### Creating `game_id` For each Unique `game_name`

ALS expects user and item identifiers as integer type, but this is missing from our dataset so, we'll need to create our own integer identifiers for the items (game_id). Pyspark already provides a `StringIndexer` class that generates label indices that are between `0` and `num_of_labels`. We will be using this to generate `game_id`.

In [114]:
indexer = StringIndexer(inputCol="game_name", outputCol="game_id")
processed_data_with_game_id = indexer.fit(processed_data).transform(processed_data)

processed_data_with_game_id.show(100)

🏃 View run trusting-dolphin-349 at: http://127.0.0.1:5000/#/experiments/1/runs/0d7d85e32c5b4b769f627bb96bd415ad
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
+---------+--------------------+------------------+-------+
|  user_id|           game_name|   relevance_score|game_id|
+---------+--------------------+------------------+-------+
|197455089|              dota_2|2.4816045409242156|    0.0|
|126340495|             outlast|2.6292405397302803|  267.0|
| 97298878|        race_the_sun|3.4423470353692043|  529.0|
| 92107940|far_cry_3_blood_d...| 4.433987204485146|  318.0|
| 11373749|     team_fortress_2| 7.637258031284457|    1.0|
| 11373749|             anodyne|3.0412203288596382| 1701.0|
| 11373749|resident_evil_5_/...|2.2237754316221157|  554.0|
| 11373749|the_movies_stunts...|               1.0| 3903.0|
| 11373749|  hitman_blood_money|               1.0|  340.0|
|171847029|arma_cold_war_ass...| 1.262364264467491|  307.0|
| 64350600|half-life_blue_shift|               1

## Modeling
___

Now that we have our processed dataset, we can begin the model training process. We split the data into training and test sets using an 80:20 ratio — a standard split that provides sufficient data for learning user and item latent factors while retaining a meaningful held-out set for evaluation. We then train the ALS model on the training set and evaluate performance on the test set.

### Split Dataset into Training and Test Set

The processed dataset is split with 80% of the data being used for training and 20% of the data kept for evaluation.

In [115]:
# Split data into training and test sets
training, test = processed_data_with_game_id.randomSplit([0.8, 0.2], seed=SEED)

### Model Training

The ALS model is trained using 3-fold cross-validation for hyperparameter tuning. Since we are working with implicit feedback (playtime and purchase behaviour rather than explicit ratings), `implicitPrefs` is set to `True`. When `implicitPrefs=True`, [Spark's ALS switches to the Weighted Regularised Matrix Factorisation](https://spark.apache.org/docs/latest/ml-collaborative-filtering.html) objective, an approach used in [Collaborative Filtering for Implicit Feedback Datasets](https://doi.org/10.1109/ICDM.2008.22). Essentially, this approach treats the data as numbers representing the strength in observations of user actions. Those numbers are then related to the level of confidence in observed user preferences, rather than explicit ratings given to items.

In this notebook, the `relevance_score` computed in the feature engineering section serves as the confidence weight — a higher score indicates stronger confidence that the user has a genuine preference for that game. The log transformation applied during feature engineering ensures these confidence weights remain well-scaled and are not dominated by extreme playtime values. 

**NOTE:** Using RMSE to evaluate model performance here isn't ideal — with `implicitPrefs=True`, the model optimises for predicting binary preferences weighted by confidence, rather than predicting the `relevance_score` 
directly. RMSE therefore does not fully reflect ranking quality. However, it will still be used as a _**proxy**_ metric for cross-validation, since `RankingEvaluator` (NDCG@K) is not directly compatible with Spark's `CrossValidator`.


**Fixed Parameters:**
| Parameter | Value | Reason |
|---|---|---|
| `userCol` | `user_id` | Unique user identifier |
| `itemCol` | `game_id` | Integer-encoded game identifier |
| `ratingCol` | `relevance_score` | Log-transformed playtime score from feature engineering |
| `coldStartStrategy` | `drop` | Avoids NaN predictions for unseen users/items |
| `implicitPrefs` | `True` | Data is implicit feedback, not explicit ratings |

**Tuning Parameters:**
| Parameter | Values | Reason |
|---|---|---|
| `rank` | `[10, 50, 100]` | Controls the number of latent factors; higher rank captures more complexity but risks overfitting |
| `regParam` | `[0.01, 0.05, 0.1]` | Regularisation strength; prevents overfitting to sparse interactions |
| `maxIter` | `[5, 10]` | Number of ALS iterations; balances convergence speed and accuracy |

In [116]:
# Instantiate ALS model
als = ALS(
    userCol="user_id",
    itemCol="game_id",
    ratingCol="relevance_score",
    coldStartStrategy="drop",
    implicitPrefs=True,
    seed=SEED
)

# Build parameter grid
param_grid = (
    ParamGridBuilder()
    .addGrid(als.rank,     [10, 50, 100])
    .addGrid(als.regParam, [0.01, 0.05, 0.1])
    .addGrid(als.maxIter,  [5, 10])
    .build()
)

# Instantiate RegressionEvaluator
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="relevance_score",
    predictionCol="prediction"
)

# Instantiate CrossValidator
cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,        # 3-fold CV
    seed=SEED
)

with mlflow.start_run(run_name="als_crossval"):
    cv_model = cv.fit(training)
    
    # Log best params
    best = cv_model.bestModel

    # Best model's RMSE on test
    predictions = best.transform(test)
    rmse = evaluator.evaluate(predictions)
    
    mlflow.log_params({
        "rank":     best.rank,
        "regParam": best._java_obj.parent().getRegParam(),
        "maxIter":  best._java_obj.parent().getMaxIter(),
        "implicitPrefs": best._java_obj.parent().getImplicitPrefs(),
        "coldStartStrategy": best._java_obj.parent().getColdStartStrategy()
    })
    mlflow.log_metric("rmse", rmse)
    mlflow.spark.log_model(best, "best_als_model")

    logger.info(f"RMSE of best model: {rmse:.4f}")


26/05/30 13:09:23 WARN DAGScheduler: Broadcasting large task binary with size 1005.7 KiB
26/05/30 13:09:24 WARN DAGScheduler: Broadcasting large task binary with size 1048.3 KiB
26/05/30 13:09:24 WARN DAGScheduler: Broadcasting large task binary with size 1007.1 KiB
26/05/30 13:09:24 WARN DAGScheduler: Broadcasting large task binary with size 1090.9 KiB
26/05/30 13:09:25 WARN DAGScheduler: Broadcasting large task binary with size 1049.7 KiB
26/05/30 13:09:25 WARN DAGScheduler: Broadcasting large task binary with size 1133.5 KiB
26/05/30 13:09:25 WARN DAGScheduler: Broadcasting large task binary with size 1092.3 KiB
26/05/30 13:09:26 WARN DAGScheduler: Broadcasting large task binary with size 1176.2 KiB
26/05/30 13:09:26 WARN DAGScheduler: Broadcasting large task binary with size 1134.9 KiB
26/05/30 13:09:26 WARN DAGScheduler: Broadcasting large task binary with size 1178.1 KiB
26/05/30 13:09:27 WARN DAGScheduler: Broadcasting large task binary with size 1135.5 KiB
26/05/30 13:09:27 WAR

🏃 View run big-cow-96 at: http://127.0.0.1:5000/#/experiments/1/runs/fab91f0c65874062961bd098641d81b7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run industrious-worm-584 at: http://127.0.0.1:5000/#/experiments/1/runs/e8f75c25a873492290aa9555bc7ef4b3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run industrious-ram-834 at: http://127.0.0.1:5000/#/experiments/1/runs/65d9e63024e74a1394193c0c4f1fff12
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run painted-ape-907 at: http://127.0.0.1:5000/#/experiments/1/runs/a5f0e372044b4c1299359010f11b9bdf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run righteous-kite-507 at: http://127.0.0.1:5000/#/experiments/1/runs/5e0ba2b272b64143a99a8da692c5ba7b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run loud-steed-956 at: http://127.0.0.1:5000/#/experiments/1/runs/402e5cf3fb1945f689bc4002468b843e
🧪 View experiment at: http://127.0.0.1:5000/#/experi

26/05/30 13:14:57 ERROR Instrumentation: org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "mlflow-artifacts"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3581)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.ml.util.FileSystemOverwrite.handleOverwrite(ReadWrite.scala:833)
	at org.apache.spark.ml.util.MLWriter.save(ReadWrite.scala:175)
	at org.apache.spark.ml.PipelineModel$PipelineModelWriter.super$save(Pipeline.scala:368)
	at org.apache.spark.ml.PipelineModel$PipelineModelWriter.$anonfun$save$4(Pipeline.scala:368)
	at org.apache.spark.ml.MLEvents.withSaveInst

🏃 View run als_crossval at: http://127.0.0.1:5000/#/experiments/1/runs/d61f1c2f945047bbbdc335f98f58d0ea
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [117]:
# Extract winning hyperparameters from the best model
best_rank     = best.rank
best_regParam = best._java_obj.parent().getRegParam()
best_maxIter  = best._java_obj.parent().getMaxIter()

logger.info(f"Best rank: {best_rank}")
logger.info(f"Best regParam: {best_regParam}")
logger.info(f"Best maxIter: {best_maxIter}")

# Show the full CV metric across all each experiment run
cv_results = list(zip(cv_model.getEstimatorParamMaps(), cv_model.avgMetrics))
cv_results.sort(key=lambda x: x[1])  # lower RMSE is better

logger.info("Top 5 configurations by mean CV RMSE:")
for params, rmse in cv_results[:5]:
    config = {p.name: v for p, v in params.items()}
    logger.info(
        f"  rank={config['rank']:>3}, "
        f"regParam={config['regParam']:<5}, "
        f"maxIter={config['maxIter']:>2}  ->  RMSE={rmse:.4f}"
    )

[2026-05-30 13:15:01.524251] INFO - Best rank: 10 (/tmp/ipykernel_805233/3871898749.py:6:<module>())
[2026-05-30 13:15:01.524840] INFO - Best regParam: 0.01 (/tmp/ipykernel_805233/3871898749.py:7:<module>())
[2026-05-30 13:15:01.525553] INFO - Best maxIter: 10 (/tmp/ipykernel_805233/3871898749.py:8:<module>())
[2026-05-30 13:15:01.526203] INFO - Top 5 configurations by mean CV RMSE: (/tmp/ipykernel_805233/3871898749.py:14:<module>())
[2026-05-30 13:15:01.526672] INFO -   rank= 10, regParam=0.01 , maxIter=10  ->  RMSE=2.4320 (/tmp/ipykernel_805233/3871898749.py:17:<module>())
[2026-05-30 13:15:01.527013] INFO -   rank= 10, regParam=0.05 , maxIter=10  ->  RMSE=2.4333 (/tmp/ipykernel_805233/3871898749.py:17:<module>())
[2026-05-30 13:15:01.527334] INFO -   rank= 10, regParam=0.1  , maxIter=10  ->  RMSE=2.4351 (/tmp/ipykernel_805233/3871898749.py:17:<module>())
[2026-05-30 13:15:01.527641] INFO -   rank= 10, regParam=0.01 , maxIter= 5  ->  RMSE=2.4481 (/tmp/ipykernel_805233/3871898749.py:1

In [118]:
# Make predictions on the test set
predictions = best.transform(test)
predictions.show()

+-------+--------------------+------------------+-------+-----------+
|user_id|           game_name|   relevance_score|game_id| prediction|
+-------+--------------------+------------------+-------+-----------+
|   5250|            ricochet|               1.0|     23|  0.6906386|
|  76767|thief_deadly_shadows|2.4816045409242156|    559|0.025139684|
|  86540|back_to_the_futur...|               1.0|    827|0.123327926|
|  86540|           far_cry_3|3.9338568698359033|    103|  0.2915623|
|  86540|      puzzle_agent_2|               1.0|   1020| 0.10447064|
|  86540|the_elder_scrolls...| 5.736198448394496|     11| 0.43460798|
| 103360|            ricochet|               1.0|     23|  0.8676318|
| 144736|       day_of_defeat|               1.0|     21|  0.6251777|
| 144736|team_fortress_cla...|               1.0|     37| 0.41165563|
| 181212|      counter-strike|2.0296194171811583|      7|  0.9258297|
| 181212|half-life_2_lost_...| 1.336472236621213|      4| 0.70273983|
| 181212|team_fortre

## Evaluation
___

While RMSE was used as the optimisation metric during cross-validation for hyperparameter tuning, it is not the most appropriate metric for evaluating a recommender system. This is because RMSE measures how accurately the model predicts the relevance score for each user-item pair, but it does not capture what we actually care about — whether the model is surfacing the most relevant games at the top of the recommendation list.

NDCG (Normalised Discounted Cumulative Gain) is the gold standard metric for evaluating ranking quality in search engines, recommendation systems, and information retrieval. This is the metric that will be used to evaluate the ranking quality of the trained model.

The key differences are summarised below:

| | RMSE | NDCG@K |
|---|---|---|
| Measures | Prediction error on observed interactions | Quality of top-K ranking |
| Penalises | Large score prediction errors | Relevant items ranked too low |
| Reflects real-world usage? | ❌ Not directly | ✅ Yes |
| Sensitive to rank position? | ❌ No | ✅ Yes |

### Normalized Discounted Cumulative Gain (NDCG)

NDCG is a ranking metric that measures the quality of the top-K recommendations by rewarding the model for placing highly relevant items higher in the list, with a logarithmic discount applied to relevant items that appear lower in the ranking.

**Discounted Cumulative Gain (DCG)** measures the total relevance of the ranked list, discounted by position:

$$\text{DCG@K} = \sum_{i=1}^{K} \frac{\text{rel}_i}{\log_2(i + 1)}$$

**Ideal DCG (IDCG)** is the best possible DCG, computed by sorting all items the user actually interacted with by their relevance score in descending order:

$$\text{IDCG@K} = \sum_{i=1}^{K} \frac{\text{rel}^*_i}{\log_2(i + 1)}$$

**NDCG** then normalises DCG against IDCG, giving a score between 0 and 1:

$$\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$

Where:
- rel_i = relevance score of the item at position _i_ in the predicted ranking
- rel_i* = relevance score of the item at position _i_ in the ideal ranking
- K = 10 (top 10 recommendations)

A score of 1.0 indicates a perfect ranking, where the most relevant items are placed at the top.

### What is a Good NDCG Score?

Given that NDCG ranges from 0 to 1, a good recommender system should have a value close to 1, right? Well, this is almost always not the case, even for big tech companies like Netflix. The problem stems from the sparse nature of datasets typically used for recommender systems. In cell 40, we calculated the sparsity of the Steam dataset and got a value of `0.9980`, meaning 99.8% of the user-item matrix is empty. This level of sparsity makes it very difficult for the model to learn patterns.

So how then can we evaluate the quality of a recommender system in the real world? By comparing it against baselines. We will compare our model against two baselines:

- **Random Baseline:** Recommends 10 games chosen at random for each user, with no personalisation. This represents the worst-case scenario and sets a lower bound — any meaningful recommender system should score significantly above this.

- **Overall Top-K Baseline:** Recommends the same top 10 most popular games (by relevance score) to every user, regardless of their individual preferences. This is a stronger baseline than random, as popular items are generally well-liked, but it has no personalisation whatsoever.

### Evaluate Model with Normalized Discounted Cumulative Gain (NDCG)

In [119]:
# Generate top 10 recommendations per user
user_recs = best.recommendForAllUsers(10)
user_recs.show(truncate=False)

+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id |recommendations                                                                                                                                                                     |
+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|835015  |[{7, 0.8381206}, {13, 0.6928802}, {21, 0.6787286}, {23, 0.66327053}, {22, 0.65875596}, {14, 0.6543406}, {32, 0.47636572}, {36, 0.46905375}, {40, 0.46781108}, {37, 0.45961022}]     |
|948368  |[{7, 0.9068959}, {21, 0.75462306}, {13, 0.75304085}, {23, 0.7382928}, {22, 0.73188376}, {14, 0.71768904}, {5, 0.66087705}, {9, 0.6529262}, {4, 0.65154165}, {32, 0.6008799}]        |
|975449  |[{51, 0.9798512}, {1, 0.950610

In [120]:
# Extract recommendations as a list of game_ids and cast to double
predictions = user_recs.select(
    "user_id",
    F.expr("transform(recommendations, x -> x.game_id)").cast("array<double>").alias("prediction")
)
predictions.show(truncate=False)

+--------+---------------------------------------------------------------+
|user_id |prediction                                                     |
+--------+---------------------------------------------------------------+
|835015  |[7.0, 13.0, 21.0, 23.0, 22.0, 14.0, 32.0, 36.0, 40.0, 37.0]    |
|948368  |[7.0, 21.0, 13.0, 23.0, 22.0, 14.0, 5.0, 9.0, 4.0, 32.0]       |
|975449  |[51.0, 1.0, 16.0, 185.0, 156.0, 168.0, 116.0, 87.0, 4.0, 250.0]|
|2753525 |[7.0, 11.0, 3.0, 5.0, 13.0, 14.0, 26.0, 20.0, 1.0, 8.0]        |
|3450426 |[7.0, 13.0, 21.0, 14.0, 23.0, 22.0, 32.0, 36.0, 40.0, 37.0]    |
|8567888 |[7.0, 5.0, 13.0, 21.0, 23.0, 14.0, 22.0, 9.0, 4.0, 32.0]       |
|8784496 |[7.0, 4.0, 21.0, 23.0, 22.0, 9.0, 16.0, 1.0, 32.0, 13.0]       |
|8795607 |[4.0, 5.0, 9.0, 7.0, 16.0, 1.0, 21.0, 13.0, 23.0, 22.0]        |
|10144413|[4.0, 5.0, 9.0, 16.0, 28.0, 19.0, 30.0, 38.0, 17.0, 61.0]      |
|11973378|[5.0, 4.0, 9.0, 3.0, 16.0, 28.0, 17.0, 19.0, 30.0, 38.0]       |
|15196746|[5.0, 4.0, 16.0

In [121]:
# Build ground truth — actual games each user interacted with
actuals = (
    test.orderBy("relevance_score", ascending=False)
    .groupBy("user_id")
    .agg(
        F.collect_list("game_id")
        .alias("label")
              )
    )
actuals.show(truncate=False)

+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|label                                                                                                                                                                                                                                                                                                                                                                                                      |
+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [122]:
# Join predictions and actuals
ranking_data = predictions.join(actuals, "user_id")

In [123]:
evaluator_ndcg = RankingEvaluator(
    predictionCol="prediction",
    labelCol="label",
    metricName="ndcgAtK",
    k=10
)

ndcg = evaluator_ndcg.evaluate(ranking_data)
logger.info(f"ndcg@10: {ndcg:.4f}")

[2026-05-30 13:15:09.060396] INFO - ndcg@10: 0.3650 (/tmp/ipykernel_805233/147302352.py:9:<module>())


### Compare Model to Random Baseline

In [124]:
# Get all distinct game_ids
all_game_ids = processed_data_with_game_id.select("game_id").distinct().collect()
all_game_ids = [row["game_id"] for row in all_game_ids]

# For each user in test set, generate 10 random game recommendations
distinct_test_users = test.select("user_id").distinct()

# UDF that returns 10 random game_ids as doubles
@F.udf("array<double>")
def random_recs(user_id):
    rng = random.Random(user_id)
    return [float(gid) for gid in rng.sample(all_game_ids, 10)]

random_predictions = distinct_test_users.select(
    "user_id",
    random_recs(F.col("user_id")).alias("prediction")
).cache()

# Join and evaluate
random_ranking_data = random_predictions.join(actuals, "user_id")

random_ndcg = evaluator_ndcg.evaluate(random_ranking_data)
logger.info(f"Random Baseline NDCG@10: {random_ndcg:.4f}")

[2026-05-30 13:15:14.017516] INFO - Random Baseline NDCG@10: 0.0018 (/tmp/ipykernel_805233/2910670680.py:23:<module>())


In [125]:
logger.info(f"Random Baseline NDCG@10: {random_ndcg:.4f}")
logger.info(f"ALS Model NDCG@10: {ndcg:.4f}")
logger.info(f"Improvement Over Random Baseline: {((ndcg - random_ndcg) / random_ndcg) * 100:.2f}%")

[2026-05-30 13:15:14.026154] INFO - Random Baseline NDCG@10: 0.0018 (/tmp/ipykernel_805233/269080109.py:1:<module>())
[2026-05-30 13:15:14.027155] INFO - ALS Model NDCG@10: 0.3650 (/tmp/ipykernel_805233/269080109.py:2:<module>())
[2026-05-30 13:15:14.027847] INFO - Improvement Over Random Baseline: 19910.52% (/tmp/ipykernel_805233/269080109.py:3:<module>())


### Compare Model to Overall Top-K Baseline

In [126]:
overall_top_10_games = (
    processed_data_with_game_id
    .select("game_id", "relevance_score")
    .groupBy("game_id")
    .agg(F.sum("relevance_score").alias("sum_relevance_score"))
    .orderBy(F.col("sum_relevance_score").desc())
    .limit(10)
    )

overall_top_10_games.show()

+-------+-------------------+
|game_id|sum_relevance_score|
+-------+-------------------+
|    0.0| 17770.228381594923|
|    1.0|  7488.587123704161|
|    3.0|    6967.4250120927|
|    2.0|  3441.349331925942|
|   11.0|  3144.588093628765|
|    5.0|  3089.855720877944|
|    6.0| 2982.9477477051455|
|   10.0| 2719.9764176738886|
|   17.0|  2576.960685994716|
|    7.0| 2497.9471559880394|
+-------+-------------------+



In [127]:
top_10_game_ids = [game[0] for game in overall_top_10_games.select("game_id").collect()]
logger.info(f"Top 10 Game IDs: {top_10_game_ids}")

[2026-05-30 13:15:14.893889] INFO - Top 10 Game IDs: [0.0, 1.0, 3.0, 2.0, 11.0, 5.0, 6.0, 10.0, 17.0, 7.0] (/tmp/ipykernel_805233/2832196038.py:2:<module>())


In [128]:
# UDF that returns 10 random game_ids as doubles
@F.udf("array<double>")
def apply_top_10(_):
    return top_10_game_ids

top_10_predictions = distinct_test_users.select(
    "user_id",
    apply_top_10(F.col("user_id")).alias("prediction")
)

# Join and evaluate
top_10_ranking_data = top_10_predictions.join(actuals, "user_id")

top_10_ndcg = evaluator_ndcg.evaluate(top_10_ranking_data)
logger.info(f"Overall Top 10 Baseline NDCG@10: {top_10_ndcg:.4f}")

[2026-05-30 13:15:15.616672] INFO - Overall Top 10 Baseline NDCG@10: 0.2257 (/tmp/ipykernel_805233/1111381665.py:15:<module>())


In [129]:
logger.info(f"Overall Top 10 Baseline NDCG@10: {top_10_ndcg:.4f}")
logger.info(f"ALS Model NDCG@10: {ndcg:.4f}")
logger.info(f"Improvement Over Top 10 Baseline: {((ndcg - top_10_ndcg) / top_10_ndcg) * 100:.2f}%")

[2026-05-30 13:15:15.622307] INFO - Overall Top 10 Baseline NDCG@10: 0.2257 (/tmp/ipykernel_805233/3511861046.py:1:<module>())
[2026-05-30 13:15:15.623241] INFO - ALS Model NDCG@10: 0.3650 (/tmp/ipykernel_805233/3511861046.py:2:<module>())
[2026-05-30 13:15:15.623669] INFO - Improvement Over Top 10 Baseline: 61.70% (/tmp/ipykernel_805233/3511861046.py:3:<module>())


### Recommend Games for Specific User(s)

In [130]:
# Build a dictionary of game_id -> game_name
item_dict = (
        processed_data_with_game_id
        .select("game_id", "game_name")
        .distinct()
        .collect()
    )
item_dict = {item["game_id"]: item["game_name"] for item in item_dict}

In [131]:
def recommend_games(user_ids, model, item_dict, top_n=10):
    """Recommend top_n games for each user in user_ids."""
    # create user dataframe
    user_df = spark.createDataFrame(user_ids, IntegerType()).toDF("user_id")

    item_df = spark.createDataFrame(item_dict.items(), schema=['game_id', 'game_name'])

    # get top_n recommendations for each user
    recommendations = (
        model
        .recommendForUserSubset(user_df, top_n)
        # Explode the recommendations to get a row per recommendation. 
        # The order of the recommendations is based on the predicted rating in descending order
        .select("user_id", F.explode("recommendations").alias("recommendation"))
        .select("user_id", "recommendation.game_id")
        # Join with item_df to get the game name
        .join(item_df, "game_id")
        .select("user_id", "game_name")
        .groupBy("user_id")
        .agg(F.collect_list("game_name").alias("recommended_games"))
    )

    return recommendations

# Get all distinct user_ids
user_ids = [2753525, 5250, 76767, 86540]

# Get top 10 recommendations for each user
top_10_predictions = recommend_games(user_ids, best, item_dict, top_n=10)
top_10_predictions.show(truncate=False)


+-------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|recommended_games                                                                                                                                                                                                                                                                                           |
+-------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2753525|[counter-strike, the_elder_scrolls_v_skyrim, counter-strik

In [132]:
# invert item dict to get game_name -> game_id
item_dict_inv = {v: k for k, v in item_dict.items()}

## Explaining Why an Item was Recommended
___
One key advantage of ALS over other recommendation approaches is its interpretability — we can explain *why* the model recommended a particular game to a particular user. 

During training, ALS learns two sets of embeddings: one for users and one for items, both living in the same latent space. Because of the **_alternating_** nature of the training process, these embeddings shape each other iteratively — item characteristics inform user taste profiles, and user taste profiles in turn refine item characteristics. This means a user's embedding is effectively a reflection of the items they have interacted with, and an item's embedding is a reflection of the users who interact with it.

By computing the dot product of the recommended item embedding with the embeddings of every item the user has interacted with, we can identify the items that made the most contributions to the recommendation.

In [133]:
def explain_recommendation(user_id, recommended_item_id, model, user_history, item_dict, top_n=10):
    """
    Explain why a particular item was recommended to a particular user.
    """

    if len(user_history) == 0:
        logger.warning("User has no history")
        return
    
    # Get the recommended item's features
    item_features = (
        model.itemFactors
        .filter(F.col("id") == recommended_item_id)
        .select("features")
        .collect()[0][0]
    )

    history_features_rows = (
        model.itemFactors
        .filter(F.col("id").isin(user_history))
        .select("id", "features")
        .collect()
    )

    # Build a dict so lookup is by id, not position
    history_feature_dict = {row["id"]: row["features"] for row in history_features_rows}

    # Calculate contributions of each history item
    contributions = [
        (game_id, np.array(history_feature_dict[game_id]).dot(item_features))
        for game_id in user_history
        if game_id in history_feature_dict
    ]

    # Sort by contribution descending
    contributions.sort(key=lambda x: x[1], reverse=True)

    logger.info(f"{item_dict.get(recommended_item_id, 'Unknown')} was recommended to user {user_id} because:")
    for game_id, score in contributions[:top_n]:
        logger.info(f" - {item_dict.get(game_id, 'Unknown')} has a contribution of {score:.4f}")



In [135]:
# Set user and recommended item
user_id = 2753525
recommended_game = "counter-strike"
recommended_item_id = item_dict_inv[recommended_game]

# Get user interaction history
user_history = (
        test
        .filter(F.col("user_id") == user_id)
        .select("game_id")
        .agg(F.collect_list("game_id").alias("game_ids"))
        .collect()[0][0]
    )

# Explain recommendation
explain_recommendation(user_id, recommended_item_id, best, user_history, item_dict)

[2026-05-30 14:13:37.684850] INFO - counter-strike was recommended to user 2753525 because: (/tmp/ipykernel_805233/901226066.py:38:explain_recommendation())
[2026-05-30 14:13:37.685429] INFO -  - counter-strike_condition_zero has a contribution of 0.2903 (/tmp/ipykernel_805233/901226066.py:40:explain_recommendation())
[2026-05-30 14:13:37.685827] INFO -  - counter-strike_condition_zero_deleted_scenes has a contribution of 0.2787 (/tmp/ipykernel_805233/901226066.py:40:explain_recommendation())
[2026-05-30 14:13:37.686240] INFO -  - ricochet has a contribution of 0.2537 (/tmp/ipykernel_805233/901226066.py:40:explain_recommendation())
[2026-05-30 14:13:37.686617] INFO -  - team_fortress_classic has a contribution of 0.1733 (/tmp/ipykernel_805233/901226066.py:40:explain_recommendation())
[2026-05-30 14:13:37.687522] INFO -  - counter-strike_global_offensive has a contribution of 0.1449 (/tmp/ipykernel_805233/901226066.py:40:explain_recommendation())
[2026-05-30 14:13:37.688003] INFO -  - c

## References and Acknowledgements
___

### References
1. Apache Spark. (2024). *Collaborative Filtering - Spark MLlib Programming Guide*. 
   https://spark.apache.org/docs/latest/ml-collaborative-filtering.html
2. DigitalSreeni. (2026). *Alternating Least Squares (ALS) - Building your 
   first production-ready recommender system* [Video]. YouTube. 
   https://www.youtube.com/watch?v=E6n0-BNsDFE